# One-hot encoding
Turning the nominal columns `fuel`, `owner` and `brand` of a used-car dataset into 0/1 columns, first with pandas, then with scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

## Load the data
8128 used cars: brand, kilometres driven, fuel type, owner, and the selling price (the target).

In [ ]:
df = pd.read_csv("data/cars.csv")
df.head()

In [ ]:
# how many cars of each fuel type and each owner type
print(df["fuel"].value_counts(), "\n")
print(df["owner"].value_counts())

In [ ]:
# brand has many categories
df["brand"].nunique()

## 1. One-hot encoding with pandas
`get_dummies` replaces each listed column by one column per category. In pandas 3 the new columns hold `True`/`False`; `dtype=int` gives 0/1.

In [ ]:
# 5 columns - 2 encoded + 4 fuel + 5 owner = 12 columns
full = pd.get_dummies(df, columns=["fuel", "owner"], dtype=int)
print(full.shape)
full.head()

## 2. Dropping the first column (n - 1 columns)
`drop_first=True` removes the first category of each column (CNG and First Owner), which avoids the dummy variable trap.

In [ ]:
k_minus_1 = pd.get_dummies(df, columns=["fuel", "owner"], drop_first=True, dtype=int)
print(k_minus_1.shape)
k_minus_1.columns.tolist()

### Why `get_dummies` is risky in an ML project
It only looks at the rows it is given. If new data holds fewer categories, we get fewer columns than the model was trained on.

In [ ]:
new_cars = pd.DataFrame({"fuel": ["Diesel", "Petrol"]})
# only 2 columns instead of 4
pd.get_dummies(new_cars, dtype=int)

## 3. One-hot encoding with scikit-learn
Split first, then let `OneHotEncoder` learn the categories from the training set.

In [ ]:
# inputs: the first 4 columns; target: selling_price
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, 0:4], df.iloc[:, -1], test_size=0.2, random_state=2)
X_train.head()

In [ ]:
# drop="first": n - 1 columns per feature
# sparse_output=False: a normal NumPy array, not a sparse matrix
# dtype=np.int32: whole numbers instead of 0.0 / 1.0
ohe = OneHotEncoder(drop="first", sparse_output=False, dtype=np.int32)

In [ ]:
# learn the categories from the training set and encode it
X_train_new = ohe.fit_transform(X_train[["fuel", "owner"]])
# encode the test set with the same categories
X_test_new = ohe.transform(X_test[["fuel", "owner"]])
X_train_new.shape, X_test_new.shape

In [ ]:
# the categories learned, and the names of the new columns
print(ohe.categories_)
ohe.get_feature_names_out()

In [ ]:
# the default (sparse_output=True) returns a sparse matrix that stores only the 1s
sparse = OneHotEncoder().fit_transform(X_train[["fuel", "owner"]])
print(repr(sparse))
# .toarray() turns it into a normal array
sparse.toarray()[:3]

### Joining the encoded columns back
`brand` and `km_driven` were not encoded, so we put them back next to the new columns. (The column transformer, next Note, does this in one step.)

In [ ]:
X_train_full = np.hstack((X_train[["brand", "km_driven"]].values, X_train_new))
print(X_train_full.shape)
X_train_full[:3]

## 4. One-hot encoding with top categories
`brand` has 32 categories. We keep brands with more than 100 cars and group the rest as `uncommon`.

In [ ]:
counts = df["brand"].value_counts()
threshold = 100
# brands with 100 cars or fewer
repl = counts[counts <= threshold].index
print(len(repl), "brands become 'uncommon'")

In [ ]:
brand_top = df["brand"].replace(repl, "uncommon")
# 12 kept brands + 1 'uncommon' = 13 columns
top = pd.get_dummies(brand_top, dtype=int)
print(top.shape)
top.sample(5, random_state=0)

### The same with `OneHotEncoder`
`min_frequency` groups rare categories during `fit`, counting in the training set only (there Skoda has 81 cars, so it is grouped too). `handle_unknown="infrequent_if_exist"` sends brands never seen before into the same group.

In [ ]:
ohe_brand = OneHotEncoder(min_frequency=101, handle_unknown="infrequent_if_exist",
                          sparse_output=False, dtype=np.int32)
ohe_brand.fit(X_train[["brand"]])
print(ohe_brand.get_feature_names_out())
# Maruti, a rare brand (Opel) and an unseen brand (Tesla)
ohe_brand.transform(pd.DataFrame({"brand": ["Maruti", "Opel", "Tesla"]}))